In [1]:
!pip install transformers torch scikit-learn

In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

In [3]:
df_train = pd.read_csv('train.csv')
df_val = pd.read_csv('val.csv')
df_test = pd.read_csv('test.csv')

label2id = {'BRCA': 0, 'LUAD': 1, 'PRAD': 2}

df_train['label'] = df_train['cancer_type'].map(label2id)
df_val['label'] = df_val['cancer_type'].map(label2id)
df_test['label'] = df_test['cancer_type'].map(label2id)

print("Train:", len(df_train), "Val:", len(df_val), "Test:", len(df_test))

Train: 1574 Val: 197 Test: 197


In [4]:
class BigBirdDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=1024):
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained('yikuan8/Clinical-BigBird')

model = AutoModelForSequenceClassification.from_pretrained(
    'yikuan8/Clinical-BigBird',
    num_labels=3
)
model = model.to(device)

print("Device:", device)
print("Model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/846k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

BigBirdForSequenceClassification LOAD REPORT from: yikuan8/Clinical-BigBird
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.out_proj.weight                 | MISSING    | 
classifier.dense.bias                      | MISSING    | 
classifier.out_proj.bias                   | MISSING    | 
classifier.dense.weight                    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you 

Device: cuda
Model loaded


In [6]:
train_dataset = BigBirdDataset(df_train, tokenizer)
val_dataset = BigBirdDataset(df_val, tokenizer)
test_dataset = BigBirdDataset(df_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

print("Train:", len(train_loader), "batches")
print("Val:", len(val_loader), "batches")
print("Test:", len(test_loader), "batches")

Train: 1574 batches
Val: 197 batches
Test: 197 batches


In [7]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [8]:
def evaluate(model, loader, device):
    model.eval()
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
    return auc

In [9]:
from torch.amp import autocast, GradScaler

model.gradient_checkpointing_enable()

scaler = GradScaler('cuda')
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 3
train_losses = []
val_aucs = []

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        with autocast('cuda'):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        torch.cuda.empty_cache()

    train_loss = total_loss / len(train_loader)
    val_auc = evaluate(model, val_loader, device)

    train_losses.append(train_loss)
    val_aucs.append(val_auc)

    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f}")
    torch.save(model.state_dict(), f'bigbird_epoch{epoch+1}.pt')
    print(f"Saved checkpoint epoch {epoch+1}")

Epoch 1 | Loss: 0.3050 | Val AUC: 0.9977
Saved checkpoint epoch 1
Epoch 2 | Loss: 0.0643 | Val AUC: 0.9989
Saved checkpoint epoch 2
Epoch 3 | Loss: 0.0493 | Val AUC: 0.9997
Saved checkpoint epoch 3


In [10]:
print(model)

BigBirdForSequenceClassification(
  (bert): BigBirdModel(
    (embeddings): BigBirdEmbeddings(
      (word_embeddings): Embedding(50358, 768, padding_idx=0)
      (position_embeddings): Embedding(4096, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BigBirdEncoder(
      (layer): ModuleList(
        (0-11): 12 x BigBirdLayer(
          (attention): BigBirdAttention(
            (self): BigBirdBlockSparseAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): BigBirdSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=Tru

In [11]:
test_auc = evaluate(model, test_loader, device)
print(f"Test AUC: {test_auc:.4f}")

Test AUC: 0.9998
